In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.9 MB/s eta 0:00:0000:01


In [2]:
import os
import sys
import math
import random
import time
import shutil
import logging
import pickle
import argparse
import numpy as np
import scipy.io as sio
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn import preprocessing, metrics
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset, Sampler

from einops import rearrange
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

In [3]:
def cost_matrix_batch_torch(x, y):
	"Returns the cosine distance batchwise"
	# x is the image feature: bs * d * m * m
	# y is the audio feature: bs * d * nF
	# return: bs * n * m
	# print(x.size())
	bs = list(x.size())[0]
	D = x.size(1)
	assert(x.size(1)==y.size(1))
	# x = x.contiguous().view(bs, D, -1) # bs * d * m^2
	x = x.div(torch.norm(x, p=2, dim=1, keepdim=True) + 1e-12)
	y = y.div(torch.norm(y, p=2, dim=1, keepdim=True) + 1e-12)
	cos_dis = torch.bmm(torch.transpose(x, 1, 2), y)#.transpose(1,2)
	cos_dis = 1 - cos_dis # to minimize this value
	# cos_dis = - cos_dis
	return cos_dis.transpose(2,1)

def cos_batch_torch(x, y):
	"Returns the cosine distance batchwise"
	# x is the image feature: bs * d * m * m
	# y is the audio feature: bs * d * nF
	# return: bs * n * m
	# print(x.size())
	bs = x.size(0)
	D = x.size(1)
	assert(x.size(1)==y.size(1))
	x = x.contiguous().view(bs, D, -1) # bs * d * m^2
	x = x.div(torch.norm(x, p=2, dim=1, keepdim=True) + 1e-12)
	y = y.div(torch.norm(y, p=2, dim=1, keepdim=True) + 1e-12)
	cos_dis = torch.bmm(torch.transpose(x,1,2), y)#.transpose(1,2)
	cos_dis = 1 - cos_dis # to minimize this value
	# return cos_dis.transpose(2,1)
	# TODO:
	beta = 0.1
	min_score = cos_dis.min()
	max_score = cos_dis.max()
	threshold = min_score + beta * (max_score - min_score)
	res = cos_dis - threshold
	# res = torch.nn.ReLU()

	return torch.nn.functional.relu(res.transpose(2,1))

def pairwise_distances(x, y=None):
	'''
	Input: x is a Nxd matrix
		   y is an optional Mxd matirx
	Output: dist is a NxM matrix where dist[i,j] is the square norm between x[i,:] and y[j,:]
			if y is not given then use 'y=x'.
	i.e. dist[i,j] = ||x[i,:]-y[j,:]||^2
	'''
	x_norm = (x ** 2).sum(1).view(-1, 1)
	if y is not None:
		y_t = torch.transpose(y, 0, 1)
		y_norm = (y ** 2).sum(1).view(1, -1)
	else:
		y_t = torch.transpose(x, 0, 1)
		y_norm = x_norm.view(1, -1)

	dist = x_norm + y_norm - 2.0 * torch.mm(x, y_t)
	# Ensure diagonal is zero if x=y
	# if y is None:
	#     dist = dist - torch.diag(dist.diag)
	return torch.clamp(dist, 0.0, np.inf)

def IPOT_distance_torch_batch_uniform(C, bs, n, m, iteration=50):
	C = C.float().cuda()
	T = IPOT_torch_batch_uniform(C  , bs, n, m, iteration=iteration)
	temp = torch.bmm(torch.transpose(C,1,2), T)
	distance = batch_trace(temp, m, bs)
	return -distance

def IPOT_torch_batch_uniform(C, bs, n, m, beta=0.5, iteration=50):
	# C is the distance matrix
	# c: bs by n by m
	sigma = torch.ones(bs, int(m), 1).cuda()/float(m)
	T = torch.ones(bs, n, m).cuda()
	A = torch.exp(-C/beta).float().cuda()
	for t in range(1):
		Q = A * T # bs * n * m
		del T
		for k in range(iteration):
			delta = 1 / (n * torch.bmm(Q, sigma))
			a = torch.bmm(torch.transpose(Q,1,2), delta)
			sigma = 1 / (float(m) * a)
		T = delta * Q * sigma.transpose(2,1)
		del Q

	return T#.detach()

def GW_distance(X, Y, p, q, lamda=0.5, iteration=5, OT_iteration=20, **kwargs):
	'''
	:param X, Y: Source and target embeddings , batchsize by embed_dim by n
	:param p, q: probability vectors
	:param lamda: regularization
	:return: GW distance
	'''
	Cs = cos_batch_torch(X, X).float().cuda()
	Ct = cos_batch_torch(Y, Y).float().cuda()

	# pdb.set_trace()
	bs = Cs.size(0)
	m = Ct.size(2)
	n = Cs.size(2)
	T, Cst = GW_torch_batch(Cs, Ct, bs, n, m, p, q, beta=lamda, iteration=iteration, OT_iteration=OT_iteration)
	temp = torch.bmm(torch.transpose(Cst,1,2), T)
	distance = batch_trace(temp, m, bs)
	return distance

def GW_torch_batch(Cs, Ct, bs, n, m, p, q, beta=0.5, iteration=5, OT_iteration=20):
	one_m = torch.ones(bs, m, 1).float().cuda()
	one_n = torch.ones(bs, n, 1).float().cuda()

	Cst = torch.bmm(torch.bmm(Cs**2, p), torch.transpose(one_m, 1, 2)) + \
	      torch.bmm(one_n, torch.bmm(torch.transpose(q,1,2), torch.transpose(Ct**2, 1, 2))) # bs by n by m
	gamma = torch.bmm(p, q.transpose(2,1)) # outer product, init
	# gamma = torch.einsum('bi,bj->bij', (torch.squeeze(p), torch.squeeze(q))) # outer product, initialization
	for i in range(iteration):
		C_gamma = Cst - 2 * torch.bmm(torch.bmm(Cs, gamma), torch.transpose(Ct, 1, 2))
		# # Sinkhorn iteration
		# b = torch.ones(bs, m, 1).cuda()
		# K = torch.exp(-C_gamma/beta)
		# for i in range(50):cd
		# 	a = p/(torch.bmm(K, b))
		# 	b = q/torch.bmm(K.transpose(1,2), a)
		# gamma = a * K * b
		gamma = IPOT_torch_batch_uniform(C_gamma, bs, n, m, beta=beta, iteration=OT_iteration)
	Cgamma = Cst - 2 * torch.bmm(torch.bmm(Cs, gamma), torch.transpose(Ct, 1, 2))
	return gamma.detach(), Cgamma

def GW_distance_uniform(X, Y, lamda=1e-1, iteration=5, OT_iteration=20, **kwargs):
	m = X.size(2)
	n = Y.size(2)
	bs = X.size(0)
	p = (torch.ones(bs, m, 1)/m).cuda()
	q = (torch.ones(bs, n, 1)/n).cuda()
	return GW_distance(X, Y, p, q, lamda=lamda, iteration=iteration, OT_iteration=OT_iteration, **kwargs)

def batch_trace(input_matrix, n, bs):
	a = torch.eye(n).cuda().unsqueeze(0).repeat(bs, 1, 1)
	b = a * input_matrix
	return torch.sum(torch.sum(b,-1),-1).unsqueeze(1)



In [4]:
# from TSTOT_torch_ import cost_matrix_batch_torch, GW_distance_uniform, IPOT_distance_torch_batch_uniform


def same_seeds(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)  # Numpy module.
    random.seed(seed)  # Python random module.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * sp.stats.t._ppf((1 + confidence) / 2., n - 1)
    return m, h


from operator import truediv


def AA_andEachClassAccuracy(confusion_matrix):
    counter = confusion_matrix.shape[0]
    list_diag = np.diag(confusion_matrix)
    list_raw_sum = np.sum(confusion_matrix, axis=1)
    each_acc = np.nan_to_num(truediv(list_diag, list_raw_sum))
    average_acc = np.mean(each_acc)
    return each_acc, average_acc


import torch.utils.data as data


class matcifar(data.Dataset):
    """`CIFAR10 <https://www.cs.toronto.edu/~kriz/cifar.html>`_ Dataset.
    Args:
        root (string): Root directory of dataset where directory
            ``cifar-10-batches-py`` exists.
        train (bool, optional): If True, creates dataset from training set, otherwise
            creates from test set.
        transform (callable, optional): A function/transform that  takes in an PIL image
            and returns a transformed version. E.g, ``transforms.RandomCrop``
        target_transform (callable, optional): A function/transform that takes in the
            target and transforms it.
        download (bool, optional): If true, downloads the dataset from the internet and
            puts it in root directory. If dataset is already downloaded, it is not
            downloaded again.
    """

    def __init__(self, imdb, train, d, medicinal):

        self.train = train  # training set or test set
        self.imdb = imdb
        self.d = d
        self.x1 = np.argwhere(self.imdb['set'] == 1)
        self.x2 = np.argwhere(self.imdb['set'] == 3)
        self.x1 = self.x1.flatten()
        self.x2 = self.x2.flatten()
        #        if medicinal==4 and d==2:
        #            self.train_data=self.imdb['data'][self.x1,:]
        #            self.train_labels=self.imdb['Labels'][self.x1]
        #            self.test_data=self.imdb['data'][self.x2,:]
        #            self.test_labels=self.imdb['Labels'][self.x2]

        if medicinal == 1:
            self.train_data = self.imdb['data'][self.x1, :, :, :]
            self.train_labels = self.imdb['Labels'][self.x1]
            self.test_data = self.imdb['data'][self.x2, :, :, :]
            self.test_labels = self.imdb['Labels'][self.x2]

        else:
            self.train_data = self.imdb['data'][:, :, :, self.x1]
            self.train_labels = self.imdb['Labels'][self.x1]
            self.test_data = self.imdb['data'][:, :, :, self.x2]
            self.test_labels = self.imdb['Labels'][self.x2]
            if self.d == 3:
                self.train_data = self.train_data.transpose((3, 2, 0, 1))  ##(17, 17, 200, 10249)
                self.test_data = self.test_data.transpose((3, 2, 0, 1))
            else:
                self.train_data = self.train_data.transpose((3, 0, 2, 1))
                self.test_data = self.test_data.transpose((3, 0, 2, 1))

    def __getitem__(self, index):
        """
        Args:
            index (int): Index
        Returns:
            tuple: (image, target) where target is index of the target class.
        """
        if self.train:

            img, target = self.train_data[index], self.train_labels[index]
        else:

            img, target = self.test_data[index], self.test_labels[index]

        return img, target

    def __len__(self):
        if self.train:
            return len(self.train_data)
        else:
            return len(self.test_data)




def sanity_check500(all_set):
    nclass = 0
    nsamples = 0
    all_good = {}
    for class_ in all_set:
        if len(all_set[class_]) >= 500:
            all_good[class_] = all_set[class_][:500]
            nclass += 1
            nsamples += len(all_good[class_])
    print('the number of class:', nclass)  ##2
    print('the number of sample:', nsamples)  ##400
    return all_good


def sanity_check_unlabel(all_set, num_unlabel):
    nclass = 0
    nsamples = 0
    all_good = {}
    for class_ in all_set:
        all_good[class_] = all_set[class_][len(all_set[class_]) - num_unlabel:]
        nclass += 1
        nsamples += len(all_good[class_])
    print('the number of class:', nclass)
    print('the number of sample:', nsamples)
    return all_good


def flip(data):
    y_4 = np.zeros_like(data)
    y_1 = y_4
    y_2 = y_4
    first = np.concatenate((y_1, y_2, y_1), axis=1)
    second = np.concatenate((y_4, data, y_4), axis=1)
    third = first
    Data = np.concatenate((first, second, third), axis=0)
    return Data


def load_data(image_file, label_file):
    image_data = sio.loadmat(image_file)
    label_data = sio.loadmat(label_file)

    data_key = 'HypeRvieW'
    label_key = 'HypeRvieW'
    data_all = image_data[data_key]
    GroundTruth = label_data[label_key]

    [nRow, nColumn, nBand] = data_all.shape
    print(data_key, nRow, nColumn, nBand)

    data = data_all.reshape(np.prod(data_all.shape[:2]), np.prod(data_all.shape[2:]))

    data_scaler = preprocessing.scale(data.astype(float))  # (X-X_mean)/X_std,
    Data_Band_Scaler = data_scaler.reshape(data_all.shape[0], data_all.shape[1], data_all.shape[2])

    return Data_Band_Scaler, GroundTruth



def radiation_noise(data, alpha_range=(0.9, 1.1), beta=1 / 25):
    alpha = np.random.uniform(*alpha_range)
    noise = np.random.normal(loc=0., scale=1.0, size=data.shape)
    return alpha * data + beta * noise


def flip_augmentation(data):
    horizontal = np.random.random() > 0.5
    vertical = np.random.random() > 0.5
    if horizontal:
        data = np.fliplr(data)
    if vertical:
        data = np.flipud(data)
    return data


class Task(object):

    def __init__(self, data, num_classes, shot_num, query_num):
        self.data = data
        self.num_classes = num_classes
        self.support_num = shot_num
        self.query_num = query_num

        class_folders = sorted(list(data))

        class_list = random.sample(class_folders, self.num_classes)

        labels = np.array(range(len(class_list)))

        labels = dict(zip(class_list, labels))

        samples = dict()

        self.support_datas = []
        self.query_datas = []
        self.support_labels = []
        self.query_labels = []
        for c in class_list:
            temp = self.data[c]  # list
            samples[c] = random.sample(temp, len(temp))
            random.shuffle(samples[c])

            self.support_datas += samples[c][:shot_num]
            self.query_datas += samples[c][shot_num:shot_num + query_num]

            self.support_labels += [labels[c] for i in range(shot_num)]
            self.query_labels += [labels[c] for i in range(query_num)]
            # print(self.support_labels)
            # print(self.query_labels)


class FewShotDataset(Dataset):
    def __init__(self, task, split='train'):
        self.task = task
        self.split = split
        self.image_datas = self.task.support_datas if self.split == 'train' else self.task.query_datas
        self.labels = self.task.support_labels if self.split == 'train' else self.task.query_labels

    def __len__(self):
        return len(self.image_datas)

    def __getitem__(self, idx):
        raise NotImplementedError("This is an abstract class. Subclass this class for your particular dataset.")


class HBKC_dataset(FewShotDataset):
    def __init__(self, *args, **kwargs):
        super(HBKC_dataset, self).__init__(*args, **kwargs)

    def __getitem__(self, idx):
        image = self.image_datas[idx]
        label = self.labels[idx]
        return image, label


# Sampler
class ClassBalancedSampler(Sampler):
    ''' Samples 'num_inst' examples each from 'num_cl' pool of examples of size 'num_per_class' '''

    def __init__(self, num_per_class, num_cl, num_inst, shuffle=True):
        self.num_per_class = num_per_class
        self.num_cl = num_cl
        self.num_inst = num_inst
        self.shuffle = shuffle

    def __iter__(self):
        # return a single list of indices, assuming that items will be grouped by class
        if self.shuffle:
            batch = [[i + j * self.num_inst for i in torch.randperm(self.num_inst)[:self.num_per_class]] for j in
                     range(self.num_cl)]
        else:
            batch = [[i + j * self.num_inst for i in range(self.num_inst)[:self.num_per_class]] for j in
                     range(self.num_cl)]
        batch = [item for sublist in batch for item in sublist]

        if self.shuffle:
            random.shuffle(batch)
        return iter(batch)

    def __len__(self):
        return 1


# dataloader
def get_HBKC_data_loader(task, num_per_class=1, split='train', shuffle=False):

    dataset = HBKC_dataset(task, split=split)

    if split == 'train':
        sampler = ClassBalancedSampler(num_per_class, task.num_classes, task.support_num,
                                       shuffle=shuffle)  # support set
    else:
        sampler = ClassBalancedSampler(num_per_class, task.num_classes, task.query_num, shuffle=shuffle)  # query set

    loader = DataLoader(dataset, batch_size=num_per_class * task.num_classes, sampler=sampler)

    return loader


def classification_map(map, groundTruth, dpi, savePath):
    fig = plt.figure(frameon=False)
    fig.set_size_inches(groundTruth.shape[1] * 2.0 / dpi, groundTruth.shape[0] * 2.0 / dpi)

    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    ax.xaxis.set_visible(False)
    ax.yaxis.set_visible(False)
    fig.add_axes(ax)

    ax.imshow(map)
    fig.savefig(savePath, dpi=dpi)

    return 0


def allocate_tensors():
    """
    init data tensors
    :return: data tensors
    """
    tensors = dict()
    tensors['support_data'] = torch.FloatTensor()  ##将list、numpy转换成tensor
    tensors['support_label'] = torch.LongTensor()
    tensors['query_data'] = torch.FloatTensor()
    tensors['query_label'] = torch.LongTensor()
    return tensors


def allocate_tensors_unlabel():
    """
    init data tensors
    :return: data tensors
    """
    tensors = dict()
    tensors['support_data'] = torch.FloatTensor()
    tensors['support_label'] = torch.LongTensor()
    tensors['query_data'] = torch.FloatTensor()
    # tensors['query_label'] = torch.LongTensor()
    return tensors


def set_tensors(tensors, batch):
    """
    set data to initialized tensors
    :param tensors: initialized data tensors
    :param batch: current batch of data
    :return: None
    """
    support_data, support_label, query_data, query_label = batch
    tensors['support_data'].resize_(support_data.size()).copy_(support_data)
    tensors['support_label'].resize_(support_label.size()).copy_(support_label)
    tensors['query_data'].resize_(query_data.size()).copy_(query_data)
    tensors['query_label'].resize_(query_label.size()).copy_(query_label)


def set_tensors_unlabel(tensors, batch):
    support_data, support_label, query_data = batch
    tensors['support_data'].resize_(support_data.size()).copy_(support_data)
    tensors['support_label'].resize_(support_label.size()).copy_(support_label)
    tensors['query_data'].resize_(query_data.size()).copy_(query_data)
    # tensors['query_label'].resize_(query_label.size()).copy_(query_label)


def set_logging_config(logdir):
    """
    set logging configuration
    :param logdir: directory put logs
    :return: None
    """
    if not os.path.exists(logdir):
        os.makedirs(logdir)
    logging.basicConfig(format="[%(asctime)s] [%(name)s] %(message)s",
                        level=logging.INFO,
                        handlers=[logging.FileHandler(os.path.join(logdir, 'log.txt')),
                                  logging.StreamHandler(os.sys.stdout)])


def save_checkpoint(state, is_best, exp_name):
    """
    save the checkpoint during training stage
    :param state: content to be saved
    :param is_best: if DPGN model's performance is the best at current step
    :param exp_name: experiment name
    :return: None
    """
    torch.save(state, os.path.join('{}'.format(exp_name), 'checkpoint.pth.tar'))
    if is_best:
        shutil.copyfile(os.path.join('{}'.format(exp_name), 'checkpoint.pth.tar'),
                        os.path.join('{}'.format(exp_name), 'model_best.pth.tar'))


def adjust_learning_rate(optimizers, lr, iteration, dec_lr_step, lr_adj_base):
    """
    adjust learning rate after some iterations
    :param optimizers: the optimizers
    :param lr: learning rate
    :param iteration: current iteration
    :param dec_lr_step: decrease learning rate in how many step
    :return: None
    """
    # new_lr = lr * (lr_adj_base ** (int(iteration / dec_lr_step)))
    new_lr = lr / math.pow((1 + 10 * (iteration - 1) / dec_lr_step), 0.75)
    for optimizer in optimizers:
        for param_group in optimizer.param_groups:
            param_group['lr'] = new_lr



def label2edge(label, device):
    """
    convert ground truth labels into ground truth edges
    :param label: ground truth labels
    :param device: the gpu device that holds the ground truth edges
    :return: ground truth edges
    """
    # get size
    num_samples = label.size(1)  ##2 ##40
    # reshape
    label_i = label.unsqueeze(-1).repeat(1, 1, num_samples)
    label_j = label_i.transpose(1, 2)
    # compute edge
    edge = torch.eq(label_i, label_j).float().to(device)
    return edge


def one_hot_encode(num_classes, class_idx, device):
    """
    one-hot encode the ground truth
    :param num_classes: number of total class
    :param class_idx: belonging class's index
    :param device: the gpu device that holds the one-hot encoded ground truth label
    :return: one-hot encoded ground truth label
    """
    return torch.eye(num_classes)[class_idx].to(device)


def preprocess(num_ways, num_shots, num_queries, batch_size, device):
    """
    prepare for train and evaluation
    :param num_ways: number of classes for each few-shot task
    :param num_shots: number of samples for each class in few-shot task
    :param num_queries: number of queries for each class in few-shot task
    :param batch_size: how many tasks per batch
    :param device: the gpu device that holds all data
    :return: number of samples in support set
             number of total samples (support and query set)
             mask for edges connect query nodes
             mask for unlabeled data (for semi-supervised setting)
    """
    # set size of support set, query set and total number of data in single task
    num_supports = num_ways * num_shots  ##2*1=2
    num_samples = num_supports + num_queries * num_ways  ##2+19*2=40

    # set edge mask (to distinguish support and query edges)
    support_edge_mask = torch.zeros(batch_size, num_samples, num_samples).to(
        device)
    support_edge_mask[:, :num_supports, :num_supports] = 1
    query_edge_mask = 1 - support_edge_mask
    evaluation_mask = torch.ones(batch_size, num_samples, num_samples).to(device)

    return num_supports, num_samples, query_edge_mask, evaluation_mask


def preprocess_one(num_supports, num_samples, batch_size, device):
    """
    prepare for train and evaluation
    :param num_ways: number of classes for each few-shot task
    :param num_shots: number of samples for each class in few-shot task
    :param num_queries: number of queries for each class in few-shot task
    :param batch_size: how many tasks per batch
    :param device: the gpu device that holds all data
    :return: number of samples in support set
             number of total samples (support and query set)
             mask for edges connect query nodes
             mask for unlabeled data (for semi-supervised setting)
    """
    # set size of support set, query set and total number of data in single task

    # set edge mask (to distinguish support and query edges)
    support_edge_mask = torch.zeros(batch_size, num_samples, num_samples).to(device)
    support_edge_mask[:, :num_supports, :num_supports] = 1
    query_edge_mask = 1 - support_edge_mask
    evaluation_mask = torch.ones(batch_size, num_samples, num_samples).to(device)

    return num_supports, query_edge_mask, evaluation_mask


def initialize_nodes_edges(batch, num_supports, tensors, batch_size, num_queries, num_ways, device):
    """
    :param batch: data batch  ##tensors_src list1234
    :param num_supports: number of samples in support set
    :param tensors: initialized tensors for holding data ##四list字典
    :param batch_size: how many tasks per batch
    :param num_queries: number of samples in query set
    :param num_ways: number of classes for each few-shot task
    :param device: the gpu device that holds all data

    :return: data of support set,
             label of support set,
             data of query set,
             label of query set,
             data of support and query set,
             label of support and query set,
             initialized node features of distribution graph (Vd_(0)),
             initialized edge features of point graph (Ep_(0)),
             initialized edge_features_of distribution graph (Ed_(0))
    """
    # allocate data in this batch to specific variables
    set_tensors(tensors, batch)
    support_data = tensors['support_data'].squeeze(0)  ##[1,2,128]
    support_label = tensors['support_label'].squeeze(0)  ##[1,2]
    query_data = tensors['query_data'].squeeze(0)  ##[1,38,128]
    query_label = tensors['query_label'].squeeze(0)  ##[1,38]

    # initialize nodes of distribution graph
    node_gd_init_support = label2edge(support_label, device)
    node_gd_init_query = (torch.ones([batch_size, num_queries, num_supports]) * torch.tensor(1. / num_supports)).to(
        device)
    node_feature_gd = torch.cat([node_gd_init_support, node_gd_init_query], dim=1)

    # initialize edges of point graph
    all_data = torch.cat([support_data, query_data], 1)  ##1,40,128
    all_label = torch.cat([support_label, query_label], 1)
    all_label_in_edge = label2edge(all_label, device)
    edge_feature_gp = all_label_in_edge.clone()

    # uniform initialization for point graph's edges Ep_0
    edge_feature_gp[:, num_supports:, :num_supports] = 1. / num_supports
    edge_feature_gp[:, :num_supports, num_supports:] = 1. / num_supports
    edge_feature_gp[:, num_supports:, num_supports:] = 0
    for i in range(num_queries):
        edge_feature_gp[:, num_supports + i, num_supports + i] = 1

    # initialize edges of distribution graph (same as point graph)
    edge_feature_gd = edge_feature_gp.clone()

    return support_data, support_label, query_data, query_label, all_data, all_label_in_edge, \
           node_feature_gd, edge_feature_gp, edge_feature_gd


def unlabel2edge(data, device):
    """
    convert ground truth labels into ground truth edges
    :param label: ground truth labels
    :param device: the gpu device that holds the ground truth edges
    :return: ground truth edges
    """
    # get size
    num_samples = data.size(1)
    # reshape
    scores = torch.einsum('bhm,bmn->bhn', data, data.transpose(2, 1))
    edge = torch.nn.functional.softmax(scores, dim=-1)
    return edge


def initialize_nodes_edges_unlabel(batch, num_supports, tensors, batch_size, num_queries, num_ways, device):
    # allocate data in this batch to specific variables
    set_tensors_unlabel(tensors, batch)
    support_data = tensors['support_data'].squeeze(0)
    support_label = tensors['support_label'].squeeze(0)
    query_data = tensors['query_data'].squeeze(0)
    # query_label = tensors['query_label'].squeeze(0)

    # initialize nodes of distribution graph

    node_gd_init_support = label2edge(support_label, device)
    node_gd_init_query = (torch.ones([batch_size, num_queries, num_supports])
                          * torch.tensor(1. / num_supports)).to(device)
    node_feature_gd = torch.cat([node_gd_init_support, node_gd_init_query], dim=1)

    # initialize edges of point graph
    all_data = torch.cat([support_data, query_data], 1)
    all_label_in_edge = unlabel2edge(all_data, device)
    edge_feature_gp = all_label_in_edge.clone()

    # uniform initialization for point graph's edges
    edge_feature_gp[:, num_supports:, :num_supports] = 1. / num_supports
    edge_feature_gp[:, :num_supports, num_supports:] = 1. / num_supports
    edge_feature_gp[:, num_supports:, num_supports:] = 0
    for i in range(num_queries):
        edge_feature_gp[:, num_supports + i, num_supports + i] = 1

    # initialize edges of distribution graph (same as point graph)
    edge_feature_gd = edge_feature_gp.clone()

    return support_data, query_data, all_data, all_label_in_edge, \
           node_feature_gd, edge_feature_gp, edge_feature_gd






def TSTgai_OT(source_share, target_share):
    cos_distance = cost_matrix_batch_torch(source_share, target_share)
    cos_distance = cos_distance.transpose(1, 2)
    # TODO: GW and Gwd as graph alignment loss
    beta = 0.1
    min_score = cos_distance.min()
    max_score = cos_distance.max()
    threshold = min_score + beta * (max_score - min_score)
    cos_dist = torch.nn.functional.relu(cos_distance - threshold)
    wd = - IPOT_distance_torch_batch_uniform(cos_dist, source_share.size(0), source_share.size(2), target_share.size(2),
                                             iteration=30)
    gwd = GW_distance_uniform(source_share, target_share)
    FGW = torch.mean(wd) + torch.mean(gwd)
    return FGW







In [5]:
# get train_loader and test_loader
def get_train_test_loader(Data_Band_Scaler, GroundTruth, class_num, shot_num_per_class):
    print(Data_Band_Scaler.shape) ## (600，500，224)
    [nRow, nColumn, nBand] = Data_Band_Scaler.shape

    '''label start'''
    num_class = int(np.max(GroundTruth))
    data_band_scaler = flip(Data_Band_Scaler)
    groundtruth = flip(GroundTruth)
    del Data_Band_Scaler
    del GroundTruth

    HalfWidth = 4
    G = groundtruth[nRow - HalfWidth:2 * nRow + HalfWidth, nColumn - HalfWidth:2 * nColumn+ HalfWidth]
    data = data_band_scaler[nRow - HalfWidth:2 * nRow + HalfWidth, nColumn - HalfWidth:2 * nColumn + HalfWidth,:]

    [Row, Column] = np.nonzero(G)
    del data_band_scaler
    del groundtruth

    nSample = np.size(Row)
    print('number of sample', nSample)

    # Sampling samples
    train = {}
    test = {}
    da_train = {} # Data Augmentation
    m = int(np.max(G))  # 2
    nlabeled =TEST_LSAMPLE_NUM_PER_CLASS  ##5
    print('labeled number per class:', nlabeled)
    print((200 - nlabeled) / nlabeled + 1)
    print(math.ceil((200 - nlabeled) / nlabeled) + 1)

    for i in range(m):
        indices = [j for j, x in enumerate(Row.ravel().tolist()) if G[Row[j], Column[j]] == i + 1]
        np.random.shuffle(indices)
        nb_val = shot_num_per_class ##5
        train[i] = indices[:nb_val]
        da_train[i] = []
        for j in range(math.ceil((200 - nlabeled) / nlabeled) + 1):  ##0-40
            da_train[i] += indices[:nb_val]
        test[i] = indices[nb_val:]

    train_indices = []
    test_indices = []
    da_train_indices = []
    for i in range(m):
        train_indices += train[i]
        test_indices += test[i]
        da_train_indices += da_train[i]
    np.random.shuffle(test_indices)

    print('the number of train_indices1:', len(train_indices))  # 5
    print('the number of test_indices1:', len(test_indices))  # 9693
    print('the number of train_indices1 after data argumentation:', len(da_train_indices))  # 200
    print('labeled sample indices1:',train_indices)

    nTrain = len(train_indices) ##5
    nTest = len(test_indices) ##9693
    da_nTrain = len(da_train_indices) ##200

    imdb = {}
    imdb['data'] = np.zeros([2 * HalfWidth + 1, 2 * HalfWidth + 1, nBand, nTrain + nTest], dtype=np.float32)  # (9,9,100,n)
    imdb['Labels'] = np.zeros([nTrain + nTest], dtype=np.int64)
    imdb['set'] = np.zeros([nTrain + nTest], dtype=np.int64)

    RandPerm = train_indices + test_indices

    RandPerm = np.array(RandPerm)

    for iSample in range(nTrain + nTest):
        imdb['data'][:, :, :, iSample] = data[Row[RandPerm[iSample]] - HalfWidth:  Row[RandPerm[iSample]] + HalfWidth + 1,
                                         Column[RandPerm[iSample]] - HalfWidth: Column[RandPerm[iSample]] + HalfWidth + 1, :]
        imdb['Labels'][iSample] = G[Row[RandPerm[iSample]], Column[RandPerm[iSample]]].astype(np.int64)

    imdb['Labels'] = imdb['Labels'] - 1
    imdb['set'] = np.hstack((np.ones([nTrain]), 3 * np.ones([nTest]))).astype(np.int64)
    print('Data is OK.')


    train_dataset = matcifar(imdb, train=True, d=3, medicinal=0)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=shot_num_per_class * class_num,shuffle=False, num_workers=0)

    test_dataset = matcifar(imdb, train=False, d=3, medicinal=0)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=0)
    del test_dataset
    del imdb

    # Data Augmentation for target domain for training
    imdb_da_train = {}
    imdb_da_train['data'] = np.zeros([2 * HalfWidth + 1, 2 * HalfWidth + 1, nBand, da_nTrain],  dtype=np.float32)  # (9,9,100,n)
    imdb_da_train['Labels'] = np.zeros([da_nTrain], dtype=np.int64)
    imdb_da_train['set'] = np.zeros([da_nTrain], dtype=np.int64)

    da_RandPerm = np.array(da_train_indices)
    for iSample in range(da_nTrain):  # radiation_noise，flip_augmentation
        imdb_da_train['data'][:, :, :, iSample] = radiation_noise(data[Row[da_RandPerm[iSample]] - HalfWidth:  Row[da_RandPerm[iSample]] + HalfWidth + 1,
                                                       Column[da_RandPerm[iSample]] - HalfWidth: Column[da_RandPerm[iSample]] + HalfWidth + 1, :])
        imdb_da_train['Labels'][iSample] = G[Row[da_RandPerm[iSample]], Column[da_RandPerm[iSample]]].astype(np.int64)


    imdb_da_train['Labels'] = imdb_da_train['Labels'] - 1  # 1-16 0-15
    imdb_da_train['set'] = np.ones([da_nTrain]).astype(np.int64)
    print('imdb_da_train ok')

    return train_loader, test_loader, imdb_da_train ,G,RandPerm,Row, Column,nTrain ##RandPerm Row Column:10249 nTrain:80

def get_target_dataset(Data_Band_Scaler, GroundTruth, class_num, shot_num_per_class):
    train_loader, test_loader, imdb_da_train,G,RandPerm,Row, Column,nTrain = get_train_test_loader(Data_Band_Scaler=Data_Band_Scaler,  GroundTruth=GroundTruth, \
                                                                     class_num=class_num,shot_num_per_class=shot_num_per_class)  # 1 classes and 5 labeled samples per class
    train_datas, train_labels = train_datas, train_labels = next(iter(train_loader)) ##train_datas(80,200,9,9)
    print('train labels:', train_labels)
    print('size of train datas:', train_datas.shape)

    print(imdb_da_train.keys())
    print(imdb_da_train['data'].shape)  ## (9, 9, 198, 200)
    print(imdb_da_train['Labels'])
    del Data_Band_Scaler, GroundTruth

    # target data with data augmentation
    target_da_datas = np.transpose(imdb_da_train['data'], (3, 2, 0, 1))
    print(target_da_datas.shape)
    target_da_labels = imdb_da_train['Labels']
    print('target data augmentation label:', target_da_labels)

    # metatrain data for few-shot classification
    target_da_train_set = {}
    for class_, path in zip(target_da_labels, target_da_datas):
        if class_ not in target_da_train_set:
            target_da_train_set[class_] = []
        target_da_train_set[class_].append(path)
    target_da_metatrain_data = target_da_train_set
    print(target_da_metatrain_data.keys())

    # target domain : batch samples for domian adaptation
    print(imdb_da_train['data'].shape)
    print(imdb_da_train['Labels'])
    target_dataset = matcifar(imdb_da_train, train=True, d=3, medicinal=0)
    target_loader = torch.utils.data.DataLoader(target_dataset, batch_size=SUM_NUM_PER_CLASS*CLASS_NUM, shuffle=True, num_workers=0)
    del target_dataset


    return train_loader, test_loader, target_da_metatrain_data, target_loader,G,RandPerm,Row, Column,nTrain

class Attention(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64, dropout = 0.1):
        super().__init__()
        inner_dim = dim_head *  heads
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads
        self.scale = dim_head ** -0.5

        self.attend = nn.Softmax(dim = -1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x):
        qkv = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)  ##1,40,1024--->1,8,40,128

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        attn = self.attend(dots)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        # print('attention')
        return self.to_out(out)##1,40,160


class AttentionCro(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64, dropout = 0.1):
        super().__init__()
        inner_dim = dim_head *  heads
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads
        self.scale = dim_head ** -0.5

        self.attend = nn.Softmax(dim = -1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x1, x2):
        qkv1 = self.to_qkv(x1).chunk(3, dim = -1)
        q1, k1, v1 = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv1)  ##1,40,1024--->1,8,40,128
        qkv2 = self.to_qkv(x2).chunk(3, dim=-1)
        q2, k2, v2 = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qkv2)  ##1,40,1024--->1,8,40,128

        dots1 = torch.matmul(q2, k1.transpose(-1, -2)) * self.scale
        attn1 = self.attend(dots1)
        out1 = torch.matmul(attn1, v2)
        out1 = rearrange(out1, 'b h n d -> b n (h d)')

        return self.to_out(out1)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.1):
        super().__init__()
        # self.layers = nn.ModuleList([])
        for _ in range(depth):
          self.atten = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
          self.Drop = nn.Dropout(dropout)
          self.norm = nn.LayerNorm(dim)


    def forward(self, x):  ##1,40,1024
        for _ in range(6):
            x1 = self.atten(x) + x
            x1 = self.Drop(x1)
            x1 = self.norm(x1) + x1
            x1 = self.Drop(x1)

        return x1

class CroTransformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.1):
        super().__init__()
        for _ in range(depth):
            self.attenCro = AttentionCro(dim, heads=heads, dim_head=dim_head, dropout=dropout)
            self.Drop = nn.Dropout(dropout)
            self.norm = nn.LayerNorm(dim)

    def forward(self, x1, x2):  ##1,40,1024
        x1_2 = self.attenCro(x1, x2)
        x1_2 = self.Drop(x1_2)
        x1_2 = self.norm(x1_2) + x1_2
        x1_2 = self.Drop(x1_2)

        return x1_2

class TrFDA(nn.Module):

    def __init__(self,
                 num_classes=2,
                 dim=160,
                 depth=6,
                 heads=8,
                 mlp_dim=2048,
                 dim_head=64,
                 dropout=0.1,
                 emb_dropout=0.1):
        super().__init__()

        self.conv2d_features = nn.Sequential(
                        nn.Conv2d(in_channels=100, out_channels=64, kernel_size=(3, 3)),
                        nn.BatchNorm2d(64),
                        nn.ReLU(),
                    )

        self.Linear = nn.Linear(3136, dim)
        self.dropout = nn.Dropout(emb_dropout)

        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim, dropout)
        self.Crotransformer = CroTransformer(dim, depth, heads, dim_head, mlp_dim, dropout)

        # self.pool = pool
        self.to_latent = nn.Identity()

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )
        self.SF = nn.Softmax(dim=-1)
        self.LN = nn.Sequential(
            nn.Conv2d(in_channels=320, out_channels=160, kernel_size=(1, 1)),  ##64-->32
            # nn.Linear(320, 160),
            # nn.BatchNorm2d(160),
            # nn.Softmax(dim=0)
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x1, x2):  ## x1:source x2:target ##40,160
        x1 = x1.unsqueeze(0)  ##(1,40,160)
        x2 = x2.unsqueeze(0)  ##(1,40,160)


        x1_1 = self.transformer(x1)  ##1,40,160---->1,40,160
        x2_1 = self.transformer(x2)

        x_fussion1 = self.Crotransformer(x1,x2)


        x1_cro = x1_1 + x_fussion1
        x2_cro = x2_1 + x_fussion1
        x1_cro = self.dropout(x1_cro)
        x2_cro = self.dropout(x2_cro)

        x1_cro = self.SF(x1_cro)
        x2_cro = self.SF(x2_cro)


        return x1_1, x2_1, x1_cro, x2_cro


##Network

class Attention_F(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64, dropout = 0.1):  ##dim=1024
        super().__init__()
        inner_dim = dim_head *  heads ##8*64
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads  ##8
        self.scale = dim_head ** -0.5

        self.attend = nn.Softmax(dim = -1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x):
        qkv = self.to_qkv(x).chunk(3, dim = -1)  #
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)  ##1,40,1024--->1,8,40,128

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        attn = self.attend(dots)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        # print('attention')
        return self.to_out(out)##1,40,160


class Transformer_F(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.1):
        super().__init__()
        # self.layers = nn.ModuleList([])
        for _ in range(depth):
          self.atten = Attention_F(dim, heads=heads, dim_head=dim_head, dropout=dropout)
          self.Drop = nn.Dropout(dropout)
          self.norm = nn.LayerNorm(dim)


    def forward(self, x):  ##1,40,1024
        for _ in range(6):
          x1 = self.atten(x) + x
          x1 = self.Drop(x1)
          x1 = self.norm(x1) + x1
          x1 = self.Drop(x1)

        return x1

def conv3x3x3(in_channel, out_channel):
    layer = nn.Sequential(
        nn.Conv3d(in_channels=in_channel,out_channels=out_channel,kernel_size=3, stride=1,padding=1,bias=False),
        nn.BatchNorm3d(out_channel),
        # nn.ReLU(inplace=True)
    )
    return layer

class residual_block(nn.Module):

    def __init__(self, in_channel,out_channel): ##1 8
        super(residual_block, self).__init__()

        self.conv1 = conv3x3x3(in_channel,out_channel)
        self.conv2 = conv3x3x3(out_channel,out_channel)
        self.conv3 = conv3x3x3(out_channel,out_channel)

    def forward(self, x): #(1,1,100,9,9)
        x1 = F.relu(self.conv1(x), inplace=True) #(1,8,100,9,9)  (1,16,25,5,5)
        x2 = F.relu(self.conv2(x1), inplace=True) #(1,8,100,9,9) (1,16,25,5,5)
        x3 = self.conv3(x2) #(1,8,100,9,9) (1,16,25,5,5)

        out = F.relu(x1+x3, inplace=True) #(1,8,100,9,9)  (1,16,25,5,5)
        return out


class D_Res_3d_CNN(nn.Module): ##1 8 16
    def __init__(self, in_channel, out_channel1, out_channel2):
        super(D_Res_3d_CNN, self).__init__()

        self.block1 = residual_block(in_channel,out_channel1)
        self.maxpool1 = nn.MaxPool3d(kernel_size=(4,2,2),padding=(0,1,1),stride=(4,2,2))
        self.block2 = residual_block(out_channel1,out_channel2)
        self.maxpool2 = nn.MaxPool3d(kernel_size=(4,2,2),stride=(4,2,2), padding=(2,1,1))
        self.conv = nn.Conv3d(in_channels=out_channel2,out_channels=32,kernel_size=3, bias=False)

        self.final_feat_dim = 160


    def forward(self, x): #x:(2,100,9,9)
        x = x.unsqueeze(1) #(2,1,100,9,9)
        x = self.block1(x) #(16,8,100,9,9)
        x = self.maxpool1(x) #(1,8,25,5,5)


        x = self.block2(x) #(1,16,25,5,5)
        x = self.maxpool2(x) #(1,16,7,3,3)

        x = self.conv(x) #(1,32,5,1,1)
        return x


class Mapping(nn.Module):
    def __init__(self, in_dimension, out_dimension):  ##source 3,100
        super(Mapping, self).__init__()
        self.preconv = nn.Conv2d(in_dimension, out_dimension, 1, 1, bias=False)
        self.preconv_bn = nn.BatchNorm2d(out_dimension)

    def forward(self, x):
        x = self.preconv(x)  ##(2,100,9,9)
        x = self.preconv_bn(x) ##(2,100,9,9)
        return x

class Network(nn.Module):
    def __init__(self,
                 num_classes=2,
                 dim=160,
                 depth=6,
                 heads=8,
                 mlp_dim=2048,
                 dim_head=64,
                 dropout=0.1,
                 emb_dropout=0.1):
        super(Network, self).__init__()
        self.feature_encoder = D_Res_3d_CNN(1,8,16)  ##(2,100,9,9)-->
        self.target_mapping = Mapping(TAR_INPUT_DIMENSION, N_DIMENSION)
        self.source_mapping = Mapping(SRC_INPUT_DIMENSION, N_DIMENSION) ##3,100
        self.Transformer = Transformer_F(dim, depth, heads, dim_head, mlp_dim, dropout)
        # self.LN = nn.Linear(200,100)
        self.LN = nn.Sequential(
            nn.Conv2d(in_channels=320, out_channels=160, kernel_size=(1, 1)),  ##64-->32
            # nn.Linear(320, 160),
            # nn.BatchNorm2d(160),
            # nn.Softmax(dim=0)
        )

    def forward(self,x1, x2, domain='source', condition='train'):
        if domain == 'target':
            x1 = self.target_mapping(x1)  ##(2,100,9,9)
            x2 = self.target_mapping(x2)  ##(2,100,9,9)
        elif domain == 'source':   ## x.support (2,3,9,9)
            x1 = self.source_mapping(x1)
            x2 = self.source_mapping(x2)

        if condition =='train':
            feature1 = self.feature_encoder(x1)
            feature1 = feature1.view(feature1.shape[0], -1)  ##2,160
            feature2 = self.feature_encoder(x2)
            feature2 = feature2.view(feature2.shape[0], -1)  ##38，160

            feature1_0 = feature1[0:1]
            feature1_1 = feature1[1:]
            feature2_0 = feature2[:19]
            feature2_1 = feature2[19:]

            feature_first = torch.cat([feature1_0, feature2_0], dim=0)
            feature_second = torch.cat([feature1_1, feature2_1], dim=0)  ##20,160
            x_or1 = feature_first
            x_or2 = feature_second

            '''TrF start'''
            feature_first = feature_first.unsqueeze(0)  ##1,n,160
            feature_second = feature_second.unsqueeze(0)  ##1,n,160
            feature_first = self.Transformer(feature_first)  ##1,20,160
            feature_second = self.Transformer(feature_second)  ##1,20,160

            t = feature_first.shape[1]
            feature_first = feature_first.reshape(t, 160)  ##20,160
            feature_second = feature_second.reshape(t, 160)  ##20,160
            feature_first = torch.cat([feature_first, x_or1], dim=1)  ##n,160*2
            feature_second = torch.cat([feature_second, x_or2], dim=1)  ##n,160*2

            feature_first = feature_first.reshape(t, 320, 1, 1)
            feature_second = feature_second.reshape(t, 320, 1, 1)
            feature_first = self.LN(feature_first)
            feature_second = self.LN(feature_second)  ##Conv2D

            feature_first = feature_first.view(feature_first.shape[0], -1)  # (20,160)
            feature_second = feature_second.view(feature_second.shape[0], -1)  # (20,160)

            featureS_0 = feature_first[0:1]
            featureS_1 = feature_second[0:1]
            featureQ_0 = feature_first[1:]
            featureQ_1 = feature_second[1:]
            featureS = torch.cat([featureS_0, featureS_1], dim=0)
            featureQ = torch.cat([featureQ_0, featureQ_1], dim=0)

        elif condition == 'test':
            feature = self.feature_encoder(x1)  ## (n,32,5,1,1)
            feature = feature.view(feature.shape[0], -1)  ##n,160
            x_or = feature
            feature = feature.unsqueeze(0)  ##1,n,160
            # print('F_TrF start')
            feature = self.Transformer(feature)
            # print('F_TrF end')
            t = feature.shape[1]
            feature = feature.reshape(t, 160)  ##n,160
            feature = torch.cat([feature, x_or], dim=1)  ##n,160*2
            feature = feature.reshape(t, 320, 1, 1)
            feature = self.LN(feature)


            featureS = feature.view(feature.shape[0], -1)  # (1,160)
            featureQ = feature.view(feature.shape[0], -1)  # (1,160)

        return featureS, featureQ



crossEntropy = nn.CrossEntropyLoss().cuda()


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.xavier_uniform_(m.weight, gain=1)
        if m.bias is not None:
            m.bias.data.zero_()
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight, 1.0, 0.02)
        m.bias.data.zero_()
    elif classname.find('Linear') != -1:

        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            m.bias.data = torch.ones(m.bias.data.size())


def euclidean_metric(a, b):
    n = a.shape[0]
    m = b.shape[0]
    a = a.unsqueeze(1).expand(n, m, -1)
    b = b.unsqueeze(0).expand(n, m, -1)
    logits = -((a - b)**2).sum(dim=2)
    return logits

In [13]:
def zeroPadding_3D(old_matrix, pad_length, pad_depth=0):
    new_matrix = np.lib.pad(old_matrix, ((pad_length, pad_length), (pad_length, pad_length), (pad_depth, pad_depth)),
                            'constant', constant_values=0)
    return new_matrix

def indexToAssignment(index_, Row, Col, pad_length):
    new_assign = {}
    for counter, value in enumerate(index_):
        assign_0 = value // Col + pad_length
        assign_1 = value % Col + pad_length
        new_assign[counter] = [assign_0, assign_1]
    return new_assign

def selectNeighboringPatch(matrix, pos_row, pos_col, ex_len):
    selected_rows = matrix[range(pos_row - ex_len, pos_row + ex_len + 1), :]
    selected_patch = selected_rows[:, range(pos_col - ex_len, pos_col + ex_len + 1)]
    return selected_patch

def sampling(groundTruth):
    labels_loc = {}
    m = len(np.unique(groundTruth)) - (1 if 0 in groundTruth else 0)
    for i in range(1, m + 1):
        indices = [j for j, x in enumerate(groundTruth.ravel().tolist()) if x == i]
        np.random.shuffle(indices)
        labels_loc[i] = indices
    whole_indices = []
    for i in range(1, m + 1):
        whole_indices += labels_loc[i]
    np.random.shuffle(whole_indices)
    return whole_indices

def load_data(mat_file, col):
    data = sio.loadmat(mat_file)
    return data[col]

def getDataAndLabels_bay(before_file, after_file, label_file):
    data_before = load_data(before_file, 'img_2004')
    data_after = load_data(after_file, 'img_2007')
    gt = load_data(label_file, 'Ref_map_multiclass').reshape(-1)

    data_diff = data_after - data_before
    data_reshaped = data_diff.reshape(-1, data_diff.shape[-1])
    data_scaled = preprocessing.scale(data_reshaped)
    data_scaled = data_scaled.reshape(data_diff.shape)

    patch_length = 4
    padded_data = zeroPadding_3D(data_scaled, patch_length)
    np.random.seed(1334)

    whole_indices = sampling(gt)
    print("Number of total samples:", len(whole_indices))

    x = np.zeros((len(whole_indices), 2 * patch_length + 1, 2 * patch_length + 1, data_scaled.shape[-1]))
    y = gt[whole_indices] - 1  # Convert labels to 0-indexed

    Row, Col = data_scaled.shape[:2]
    assign = indexToAssignment(whole_indices, Row, Col, patch_length)

    for i in range(len(assign)):
        x[i] = selectNeighboringPatch(padded_data, assign[i][0], assign[i][1], patch_length)

    imdb = {
        'data': x.astype(np.float32),
        'Labels': y.astype(np.int64),
        'set': np.ones(len(y), dtype=np.int64)
    }
    return imdb

# Replace these with actual Bay Area paths or upload them
before_file = '/kaggle/input/d/somisettyjitendra/farmland/PreImg_2004.mat'
after_file = '/kaggle/input/d/somisettyjitendra/farmland/PostImg_2007.mat'
label_file = '/kaggle/input/d/somisettyjitendra/farmland/Reference_Map_Multiclass.mat'

imdb_bay = getDataAndLabels_bay(before_file, after_file, label_file)

# Save the new pickle file
with open('/kaggle/working//BayArea_imdb.pickle', 'wb') as handle:
    pickle.dump(imdb_bay, handle, protocol=4)

print('Bay Area data saved to: datasets/BayArea_imdb.pickle')


Number of total samples: 40500
Bay Area data saved to: datasets/BayArea_imdb.pickle


In [17]:
def load_data(image_file, label_file, col):
    image_data = sio.loadmat(image_file)
    label_data = sio.loadmat(label_file)

    data_key = col
    label_key = 'Ref_map_multiclass'
    data_all = image_data[data_key]
    GroundTruth = label_data[label_key]

    [nRow, nColumn, nBand] = data_all.shape
    print(data_key, nRow, nColumn, nBand)

    data = data_all.reshape(np.prod(data_all.shape[:2]), np.prod(data_all.shape[2:]))

    data_scaler = preprocessing.scale(data.astype(float))  # (X-X_mean)/X_std,
    Data_Band_Scaler = data_scaler.reshape(data_all.shape[0], data_all.shape[1], data_all.shape[2])

    return Data_Band_Scaler, GroundTruth

In [ ]:
parser = argparse.ArgumentParser(description="Few Shot Visual Recognition")
parser.add_argument("-f","--feature_dim",type = int, default = 160)
parser.add_argument("-c","--src_input_dim",type = int, default = 159)
parser.add_argument("-d","--tar_input_dim",type = int, default = 242)
parser.add_argument("-n","--n_dim",type = int, default = 100)
parser.add_argument("-w","--class_num",type = int, default = 2)
parser.add_argument("-s","--shot_num_per_class",type = int, default = 1)
parser.add_argument("-b","--query_num_per_class",type = int, default = 19)
parser.add_argument("-sum","--sum_num_per_class",type = int, default = 20)
parser.add_argument("-e","--episode",type = int, default= 1000)
parser.add_argument("-t","--test_episode", type = int, default = 600)
parser.add_argument("-l","--learning_rate", type = float, default = 0.001)
parser.add_argument("-g","--gpu",type=int, default=0)
parser.add_argument("-u","--hidden_unit",type=int,default=10)
parser.add_argument("-can","--hyperparameter",type=int,default=10)
# target
parser.add_argument("-m" ,"--test_class_num",type=int, default=2)
parser.add_argument("-z","--test_lsample_num_per_class",type=int,default=70, help='5 4 3 2 1')

args = parser.parse_args(args=[])

# Hyper Parameters
FEATURE_DIM = args.feature_dim  ##160
SRC_INPUT_DIMENSION = args.src_input_dim ##224
TAR_INPUT_DIMENSION = args.tar_input_dim ##224
N_DIMENSION = args.n_dim ##100
CLASS_NUM = args.class_num  ##2
SHOT_NUM_PER_CLASS = args.shot_num_per_class ##1
QUERY_NUM_PER_CLASS = args.query_num_per_class ##19
SUM_NUM_PER_CLASS = args.sum_num_per_class ##1+19
EPISODE = args.episode
TEST_EPISODE = args.test_episode ##600
LEARNING_RATE = args.learning_rate
GPU = args.gpu
HIDDEN_UNIT = args.hidden_unit ##10
CAN = args.hyperparameter
print(CAN)
# Hyper Parameters in target domain data set
TEST_CLASS_NUM = args.test_class_num # the number of class ##2
TEST_LSAMPLE_NUM_PER_CLASS = args.test_lsample_num_per_class # the number of labeled samples per class 5 4 3 2 1 ##5

patchsize = 9

same_seeds(0)
def _init_():
    if not os.path.exists('checkpoints'):
        os.makedirs('checkpoints')
    if not os.path.exists('classificationMap'):
        os.makedirs('classificationMap')
_init_()

# load source domain data set
with open('/kaggle/working//BayArea_imdb.pickle', 'rb') as handle:
    source_imdb = pickle.load(handle)
print(source_imdb.keys())
print(source_imdb['Labels'])

# process source domain data set
data_train = source_imdb['data']
labels_train = source_imdb['Labels']
keys_all_train = sorted(list(set(labels_train)))

label_encoder_train = {}
for i in range(len(keys_all_train)):
    label_encoder_train[keys_all_train[i]] = i
print(label_encoder_train)

train_set = {}
for class_, path in zip(labels_train, data_train):
    if label_encoder_train[class_] not in train_set:
        train_set[label_encoder_train[class_]] = []
    train_set[label_encoder_train[class_]].append(path)
print(train_set.keys())
data = train_set
del train_set
del keys_all_train
del label_encoder_train

print("Num classes for source domain datasets: " + str(len(data)))

data = sanity_check500(data) # 500 labels samples per class
print("Num classes of the number of class larger than 500 in dataset: " + str(len(data)))

for class_ in data:
    for i in range(len(data[class_])):
        image_transpose = np.transpose(data[class_][i], (2, 0, 1))
        data[class_][i] = image_transpose

# source few-shot classification data
metatrain_data= data
print(len(metatrain_data.keys()), metatrain_data.keys())
del data

# source domain adaptation data
print(source_imdb['data'].shape)
source_imdb['data'] = source_imdb['data'].transpose((1, 2, 3, 0))
print(source_imdb['data'].shape) #
print(source_imdb['Labels'])
source_dataset = matcifar(source_imdb, train=True, d=3, medicinal=0)
source_loader = torch.utils.data.DataLoader(source_dataset, batch_size=SUM_NUM_PER_CLASS*CLASS_NUM, shuffle=True, num_workers=0)

del source_dataset, source_imdb

# target domain data set

# load target domain data set
fileloc = '/kaggle/input/d/somisettyjitendra/farmland'
test_data1 = f'{fileloc}/PreImg_2004.mat'
test_data2 = f'{fileloc}/PostImg_2007.mat'
test_label = f'{fileloc}/Reference_Map_Multiclass.mat'


Data_Band_Scaler1,GroundTruth = load_data(test_data1, test_label, 'img_2004')
Data_Band_Scaler2,GroundTruth = load_data(test_data2, test_label, 'img_2007')
Data_Band_Scaler = Data_Band_Scaler2-Data_Band_Scaler1


nDataSet =1
acc = np.zeros([nDataSet, 1])
A = np.zeros([nDataSet, CLASS_NUM])
k = np.zeros([nDataSet, 1])
best_predict_all = []
best_acc_all = 0.0
best_kappa = 0.0
best_G,best_RandPerm,best_Row, best_Column,best_nTrain = None,None,None,None,None

##Bay seeds
seeds = [1330,1008,1334,2586,1220,1999,1224,1258,1334,1226,2586,1999,1008,3407,1335,1331,1334,1336,1229 ]
for iDataSet in range(nDataSet):

    # load target domain data for training and testing
    np.random.seed(seeds[iDataSet])
    print('seeds:', seeds[iDataSet])
    train_loader, test_loader, target_da_metatrain_data, target_loader,G,RandPerm,Row, Column,nTrain = get_target_dataset(
        Data_Band_Scaler=Data_Band_Scaler, GroundTruth=GroundTruth,class_num=TEST_CLASS_NUM, shot_num_per_class=TEST_LSAMPLE_NUM_PER_CLASS)

    # model
    feature_encoder = Network()
    DA_TrF = TrFDA()


    # total1 = sum([param.nelement() for param in feature_encoder.parameters()])
    # total2 = sum([param.nelement() for param in DA_TrF.parameters()])
    # print("Numer of parameter:%.2fM" % ((total1+total2)/1e6))


    feature_encoder.apply(weights_init)
    DA_TrF.apply(weights_init)

    feature_encoder.cuda()
    DA_TrF.cuda()

    feature_encoder.train()
    DA_TrF.train()

    # optimizer
    feature_encoder_optim = torch.optim.Adam(feature_encoder.parameters(), lr=args.learning_rate)
    DA_TrF_optim  = torch.optim.Adam(DA_TrF.parameters(), lr=args.learning_rate) ##lr=0.001

    print("Training...")

    last_accuracy = 0.0
    last_kappa = 0.0
    best_episdoe = 0
    train_loss = []
    test_acc = []
    running_D_loss, running_F_loss = 0.0, 0.0
    running_label_loss = 0
    running_domain_loss = 0
    total_hit_sor, total_num_sor, total_hit_tar, total_num_tar = 0.0, 0.0, 0.0, 0.0
    test_acc_list = []


    source_iter = iter(source_loader)
    target_iter = iter(target_loader)
    len_dataloader = min(len(source_loader), len(target_loader))
    train_start = time.time()
    for episode in range(2000):  # EPISODE = 90000
        # get domain adaptation data from  source domain and target domain
        try:
            source_data, source_label = next(source_iter)
        except Exception as err:
            source_iter = iter(source_loader)
            source_data, source_label = next(source_iter)

        try:
            target_data, target_label = next(target_iter)
        except Exception as err:
            target_iter = iter(target_loader)
            target_data, target_label = next(target_iter)


        '''Few-shot claification for source domain data set'''
        task_sor = Task(metatrain_data, CLASS_NUM, SHOT_NUM_PER_CLASS,QUERY_NUM_PER_CLASS)
        support_dataloader_sor = get_HBKC_data_loader(task_sor, num_per_class=SHOT_NUM_PER_CLASS, split="train",shuffle=False)
        query_dataloader_sor = get_HBKC_data_loader(task_sor, num_per_class=QUERY_NUM_PER_CLASS, split="test",shuffle=True)


        task_tar = Task(target_da_metatrain_data, TEST_CLASS_NUM, SHOT_NUM_PER_CLASS, QUERY_NUM_PER_CLASS)  # 1， 1，19
        support_dataloader_tar = get_HBKC_data_loader(task_tar, num_per_class=SHOT_NUM_PER_CLASS, split="train",shuffle=False)
        query_dataloader_tar = get_HBKC_data_loader(task_tar, num_per_class=QUERY_NUM_PER_CLASS, split="test",shuffle=True)


        supports_sor, support_labels_sor = next(iter(support_dataloader_sor))
        querys_sor, query_labels_sor = next(iter(query_dataloader_sor))



        supports_tar, support_labels_tar = next(iter(support_dataloader_tar))  # supports(2, 3, 9, 9)
        querys_tar, query_labels_tar = next(iter(query_dataloader_tar))        # querys(38, 3, 9, 9)


        # calculate features
        support_features_sor,query_features_sor = feature_encoder(supports_sor.cuda(), querys_sor.cuda(), condition='train')

        support_features_tar, query_features_tar = feature_encoder(supports_tar.cuda(), querys_tar.cuda(), domain='target',condition='train')

        # Prototype network
        if SHOT_NUM_PER_CLASS > 1:
            support_proto_sor = support_features_sor.reshape(CLASS_NUM, SHOT_NUM_PER_CLASS, -1).mean(dim=1)  # (9, 160)
            support_proto_tar = support_features_tar.reshape(CLASS_NUM, SHOT_NUM_PER_CLASS, -1).mean(dim=1)  # (9, 160)
        else:
            support_proto_sor = support_features_sor
            support_proto_tar = support_features_tar

        '''few-shot learning'''
        if (episode + 1) % 2 == 0:
            logits = euclidean_metric(query_features_sor, support_proto_sor)
            f_loss_sor = crossEntropy(logits, query_labels_sor.cuda().long())
            f_loss = f_loss_sor
        else:
            logits = euclidean_metric(query_features_tar, support_proto_tar)
            f_loss_tar = crossEntropy(logits, query_labels_tar.cuda().long())
            f_loss = f_loss_tar

        '''domain adaptation'''
        DA_features_sor_f = torch.cat([support_features_sor, query_features_sor], dim=0)  ##output.cat 40,160
        DA_features_tar_f = torch.cat([support_features_tar, query_features_tar], dim=0)

        '''TrFDA'''
        DA_TrF_sor, DA_TrF_tar,DA_TrF_sor2,DA_TrF_tar2 = DA_TrF(DA_features_sor_f, DA_features_tar_f) ##(1,40,1024)


        DA_TrF_sor2 = rearrange(DA_TrF_sor2, 'b h w -> b w h')  ##1,160,40
        DA_TrF_tar2 = rearrange(DA_TrF_tar2, 'b h w -> b w h')

        '''GOT'''

        FWD2 = TSTgai_OT(DA_TrF_sor2, DA_TrF_tar2)


        loss = f_loss  + CAN*FWD2



        # Update parameters
        feature_encoder.zero_grad()
        DA_TrF_optim.zero_grad()
        loss.backward()
        DA_TrF_optim.step()
        feature_encoder_optim.step()

        total_hit_sor += torch.sum(torch.argmax(logits, dim=1).cpu() == query_labels_sor.long()).item()
        total_num_sor += querys_sor.shape[0]
        total_hit_tar += torch.sum(torch.argmax(logits, dim=1).cpu() == query_labels_tar.long()).item()
        total_num_tar += querys_tar.shape[0]


        if (episode + 1) % 10== 0:  # display
            # train_loss.append(loss.item())
            print(
                'episode {:>3d}:   fsl loss: {:6.4f},  acc_src {:6.4f}, acc_tar {:6.4f}, loss: {:6.4f}'.format(episode + 1,
                                                                                                                f_loss.item(),
                                                                                                                total_hit_sor / total_num_sor,
                                                                                                                total_hit_tar / total_num_tar,
                                                                                                                f_loss.item()))



        '''----TEST----'''
        if (episode + 1) % 100 == 0:
            # test
            print("Testing ...")
            train_end = time.time()
            feature_encoder.eval()
            total_rewards = 0
            counter = 0
            accuracies = []
            predict = np.array([], dtype=np.int64)
            labels = np.array([], dtype=np.int64)

            train_datas, train_labels = next(iter(train_loader))
            train_features, _= feature_encoder(Variable(train_datas).cuda(), Variable(train_datas).cuda(), domain='target',condition='test')  # (45, 160)

            max_value = train_features.max()
            min_value = train_features.min()
            print(max_value.item())
            print(min_value.item())

            train_features = (train_features - min_value) * 1.0 / (max_value - min_value)

            KNN_classifier = KNeighborsClassifier(n_neighbors=1)
            KNN_classifier.fit(train_features.cpu().detach().numpy(), train_labels)
            for test_datas, test_labels in test_loader:
                batch_size = test_labels.shape[0]

                test_features, _ = feature_encoder(Variable(test_datas).cuda(), Variable(test_datas).cuda(), domain='target',condition='test')
                test_features = (test_features - min_value) * 1.0 / (max_value - min_value)
                predict_labels = KNN_classifier.predict(test_features.cpu().detach().numpy())


                test_labels = test_labels.numpy()
                rewards = [1 if predict_labels[j] == test_labels[j] else 0 for j in range(batch_size)]

                total_rewards += np.sum(rewards)
                counter += batch_size

                predict = np.append(predict, predict_labels)
                labels = np.append(labels, test_labels)

                accuracy = total_rewards / 1.0 / counter
                accuracies.append(accuracy)

            test_accuracy = 100. * total_rewards / len(test_loader.dataset)

            print('\t\tAccuracy: {}/{} ({:.2f}%)\n'.format( total_rewards, len(test_loader.dataset),100. * total_rewards / len(test_loader.dataset)))
            test_end = time.time()
            print('seeds:', seeds[iDataSet])

            # Training mode
            feature_encoder.train()
            if test_accuracy > last_accuracy:
                last_accuracy = test_accuracy
                best_episdoe = episode


                acc[iDataSet] = 100. * total_rewards / len(test_loader.dataset)
                OA = acc
                C = metrics.confusion_matrix(labels, predict)
                A[iDataSet, :] = np.diag(C) / np.sum(C, 1, dtype=np.float64)
                best_predict_all = predict
                best_G, best_RandPerm, best_Row, best_Column, best_nTrain = G, RandPerm, Row, Column, nTrain
                k[iDataSet] = metrics.cohen_kappa_score(labels, predict)
                count_0=0
                count_TP=0
                count_00=0
                for jj in range(len(test_loader.dataset)):
                    if predict[jj] == 0:
                        count_0 = count_0 + 1
                    if labels[jj] ==0:
                        count_00 = count_00 + 1
                    if predict[jj]==0 and labels[jj] == 0:
                        count_TP = count_TP+1
                precision = count_TP/count_0
                recall = count_TP/count_00



            print('best episode:[{}], best accuracy={}'.format(best_episdoe + 1, last_accuracy))


    print('iter:{} best episode:[{}], best accuracy={}'.format(iDataSet, best_episdoe + 1, last_accuracy))
    print('***********************************************************************************')
    for i in range(len(best_predict_all)):
        best_G[best_Row[best_RandPerm[best_nTrain + i]]][best_Column[best_RandPerm[best_nTrain + i]]] = best_predict_all[i] + 1


AA = np.mean(A, 1)

AAMean = np.mean(AA,0)
AAStd = np.std(AA)

AMean = np.mean(A, 0)
AStd = np.std(A, 0)

OAMean = np.mean(acc)
OAStd = np.std(acc)

kMean = np.mean(k)
kStd = np.std(k)
F1 = 2*(precision*recall)/(precision+recall)

print ("train time per DataSet(s): " + "{:.5f}".format(train_end-train_start))
print("test time per DataSet(s): " + "{:.5f}".format(test_end-train_end))
print ("average OA: " + "{:.2f}".format( OAMean) + " +- " + "{:.2f}".format( OAStd))
print ("average AA: " + "{:.2f}".format(100 * AAMean) + " +- " + "{:.2f}".format(100 * AAStd))
print ("average kappa: " + "{:.4f}".format(100 *kMean) + " +- " + "{:.4f}".format(100 *kStd))
print ("precision: " + "{:.4f}".format(100 *precision) + " +- " + "{:.4f}".format(100 *precision))
print ("recall: " + "{:.4f}".format(100 *recall) + " +- " + "{:.4f}".format(100 *recall))
print ("F1: " + "{:.4f}".format(100 *F1) + " +- " + "{:.4f}".format(100 *F1))

print ("accuracy for each class: ")
for i in range(CLASS_NUM):
    print ("Class " + str(i) + ": " + "{:.2f}".format(100 * AMean[i]) + " +- " + "{:.2f}".format(100 * AStd[i]))


best_iDataset = 0
for i in range(len(acc)):
    print('{}:{}'.format(i, acc[i]))
    if acc[i] > acc[best_iDataset]:
        best_iDataset = i
print('best acc all={}'.format(acc[best_iDataset]))

#################classification map################################

#Bay Santa
hsi_pic = np.zeros((best_G.shape[0], best_G.shape[1], 3))
for i in range(best_G.shape[0]):
    for j in range(best_G.shape[1]):
        if best_G[i][j] == 0:
            hsi_pic[i, j, :] = [0, 0, 0]
        if best_G[i][j] == 1:
            hsi_pic[i, j, :] = [0.5, 0.5, 0.5]
        if best_G[i][j] == 2:
            hsi_pic[i, j, :] = [1, 1, 1]



utils.classification_map(hsi_pic[4:-4, 4:-4, :], best_G[4:-4, 4:-4], 24,  "classificationMap/CSIDC_Bay_{}shot.png".format(TEST_LSAMPLE_NUM_PER_CLASS))
# /kaggle/working//BayArea_imdb.pickle

10
dict_keys(['data', 'Labels', 'set'])
[6 6 6 ... 6 6 6]
{0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6}
dict_keys([6, 2, 0, 3, 1, 5, 4])
Num classes for source domain datasets: 7
the number of class: 6
the number of sample: 3000
Num classes of the number of class larger than 500 in dataset: 6
6 dict_keys([6, 2, 0, 3, 1, 5])
(40500, 9, 9, 159)
(9, 9, 159, 40500)
[6 6 6 ... 6 6 6]
img_2004 225 180 159
img_2007 225 180 159
seeds: 1330
(225, 180, 159)
number of sample 40500
labeled number per class: 70
2.857142857142857
3
the number of train_indices1: 490
the number of test_indices1: 40010
the number of train_indices1 after data argumentation: 1470
labeled sample indices1: [34794, 37999, 34069, 39112, 36932, 38022, 39462, 39786, 39279, 38014, 34787, 39429, 37670, 38887, 39070, 34783, 38740, 40187, 33697, 37857, 39793, 39832, 33343, 38561, 40007, 32799, 37834, 39065, 37299, 34604, 38931, 33520, 38180, 33698, 34973, 40370, 38036, 35153, 38752, 37116, 40156, 33156, 34422, 36757, 38748, 38538, 38

img_2007: shape = (225, 180, 159) → Rows = 225, Cols = 180


Ref_map_multiclass: shape = (225, 180) → Rows = 225, Cols = 180
